# 🌸 Blue Box Color Studio — Google Colab Cloud Colorization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/margacis2/manga_colour/blob/main/notebooks/colab_colorization.ipynb)

This notebook allows you to run the **Blue Box Color Studio AI Manga Colorization Pipeline** for free using Google Colab's cloud T4 GPU acceleration.

### Features:
- **5-Channel ONNX Neural Colorization**: Uses `manga-colorize-fp16.onnx`.
- **Luminance-Preserving CIELAB Transfer**: Preserves fine ink lines, screentones, and speech bubbles.
- **Single Page & Batch Processing**: Colorize individual pages or entire chapters (.zip / .cbz).
- **Character Color Palettes**: Apply consistent character colors.

### 1. Check Hardware & Setup Environment

In [ ]:
!nvidia-smi

# Clone the repository
!git clone https://github.com/margacis2/manga_colour.git
%cd manga_colour

# Install dependencies
!pip install -q opencv-python-headless pillow numpy pypdf matplotlib

### 2. Download Model Weights
Ensure `manga-colorize-fp16.onnx` is present in `data/models/colorization/`.

In [ ]:
import os
from pathlib import Path

model_dir = Path("data/models/colorization")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "manga-colorize-fp16.onnx"

if not model_path.exists():
    print("Downloading pretrained colorization model weights...")
    # Direct model download URL fallback
    !curl -L -o "{model_path}" "https://github.com/margacis2/manga_colour/releases/download/v1.0.0/manga-colorize-fp16.onnx" || echo "Please place manga-colorize-fp16.onnx into data/models/colorization/"
else:
    print("Model weights found at:", model_path)

### 3. Load Colorization Engine

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def get_colorization_net(model_file="data/models/colorization/manga-colorize-fp16.onnx"):
    net = cv2.dnn.readNetFromONNX(model_file)
    # Try CUDA acceleration if available in OpenCV DNN
    try:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA_FP16)
    except Exception:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    return net

def colorize_manga(image_path, net):
    orig_bgr = cv2.imread(str(image_path))
    if orig_bgr is None:
        raise ValueError(f"Could not load {image_path}")
    
    orig_h, orig_w = orig_bgr.shape[:2]
    orig_gray = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2GRAY)
    
    max_dim = 1024
    scale = min(1.0, max_dim / max(orig_h, orig_w))
    target_w = max(64, (int(orig_w * scale) // 32) * 32)
    target_h = max(64, (int(orig_h * scale) // 32) * 32)
    
    gray_resized = cv2.resize(orig_gray, (target_w, target_h), interpolation=cv2.INTER_AREA)
    gray_norm = gray_resized.astype(np.float32) / 255.0
    
    inp = np.zeros((1, 5, target_h, target_w), dtype=np.float32)
    inp[0, 0] = gray_norm
    
    net.setInput(inp, "input")
    out = net.forward("rgb")[0]
    
    model_rgb = np.transpose(out, (1, 2, 0))
    model_rgb = np.clip(model_rgb * 255.0, 0, 255).astype(np.uint8)
    color_full = cv2.resize(model_rgb, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)
    
    # CIELAB luminance transfer: preserve 100% original ink lines
    color_lab = cv2.cvtColor(color_full, cv2.COLOR_RGB2LAB)
    color_lab[:, :, 0] = orig_gray
    final_rgb = cv2.cvtColor(color_lab, cv2.COLOR_LAB2RGB)
    
    return final_rgb

net = get_colorization_net()
print("Engine initialized successfully!")

### 4. Upload & Colorize Manga Pages

In [ ]:
from google.colab import files

print("Upload your manga page (PNG or JPG):")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f"Colorizing {fn}...")
    result = colorize_manga(fn, net)
    
    # Display side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(14, 10))
    orig = Image.open(fn)
    axes[0].imshow(orig, cmap='gray')
    axes[0].set_title("Original B&W Scan")
    axes[0].axis('off')
    
    axes[1].imshow(result)
    axes[1].set_title("AI Colorized Output")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    
    out_name = f"colorized_{fn}"
    Image.fromarray(result).save(out_name)
    files.download(out_name)